In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import time
from scipy.sparse import csr_matrix

# Data

In [ ]:
interactions = pd.read_csv("https://raw.githubusercontent.com/MiraFedo/Machine-Learning-/main/interactions_train.csv")
interactions = interactions.rename(columns={'u': 'user_id', 'i': 'book_id', 't': 'timestamp'})
books = pd.read_csv("https://raw.githubusercontent.com/MiraFedo/Machine-Learning-/main/items.csv")

# Load classified books and merge
books_classified = pd.read_csv("https://raw.githubusercontent.com/MiraFedo/Machine-Learning-/main/books_classified.csv")
books = books.merge(books_classified[['i', 'book_type', 'discipline', 'topic', 'confidence']],
                    on='i', how='left')

user_id_map = {orig: new for new, orig in enumerate(interactions['user_id'].unique())}
book_id_map = {orig: new for new, orig in enumerate(interactions['book_id'].unique())}
user_id_inverse_map = {v: k for k, v in user_id_map.items()}
book_id_inverse_map = {v: k for k, v in book_id_map.items()}

interactions['user_id'] = interactions['user_id'].map(user_id_map)
interactions['book_id'] = interactions['book_id'].map(book_id_map)

n_users = interactions['user_id'].nunique()
n_items = interactions['book_id'].nunique()
print(f"Users: {n_users}, Items: {n_items}")
print(f"Books with categories: {books['book_type'].notna().sum()}")

Users: 7838, Items: 15109
Books with categories: 15291


# Functions with temporal weighting:

In [ ]:
def create_data_matrix(data, n_users, n_items):
    data_matrix = np.zeros((n_users, n_items))
    data_matrix[data["user_id"].values, data["book_id"].values] = 1
    return data_matrix

def create_weighted_matrix(data, n_users, n_items, decay=0.03):
    data_matrix = np.zeros((n_users, n_items))
    for user_id, user_data in data.groupby('user_id'):
        user_data = user_data.sort_values('timestamp')
        n = len(user_data)
        weights = np.array([(1 - decay) ** (n - 1 - i) for i in range(n)])
        if weights.max() > 0:
            weights = weights / weights.max()
        for i, (_, row) in enumerate(user_data.iterrows()):
            data_matrix[int(row['user_id']), int(row['book_id'])] = weights[i]
    return data_matrix

def user_based_predict(interactions, similarity, epsilon=1e-9):
    pred = similarity.dot(interactions) / (np.abs(similarity).sum(axis=1)[:, np.newaxis] + epsilon)
    return pred

def item_based_predict(interactions, similarity, epsilon=1e-9):
    pred = similarity.dot(interactions.T) / (similarity.sum(axis=1)[:, np.newaxis] + epsilon)
    return pred.T

def precision_recall_at_k(prediction, ground_truth, k=10):
    num_users = prediction.shape[0]
    precision_at_k, recall_at_k = 0, 0
    for user in range(num_users):
        top_k_items = np.argsort(prediction[user, :])[-k:]
        relevant = np.isin(top_k_items, np.where(ground_truth[user, :] == 1)[0]).sum()
        total = ground_truth[user, :].sum()
        precision_at_k += relevant / k
        recall_at_k += relevant / total if total > 0 else 0
    return precision_at_k / num_users, recall_at_k / num_users

def normalize_matrix(matrix):
    min_val = matrix.min()
    max_val = matrix.max()
    return (matrix - min_val) / (max_val - min_val + 1e-9)

print("Functions ready!")

Functions ready!


In [ ]:
from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import hstack, csr_matrix

# 1. Preprocessing: Combine discipline/topic
books_feat = books.copy()
# We treat the combination as a single categorical feature string
books_feat['combined_cat'] = books_feat['discipline'].fillna('none') + "_" + books_feat['topic'].fillna('none')

# 2. One-hot encode the combined categories
cat_encoder = OneHotEncoder()
# Reshape to 2D array as required by encoder
cats_sparse = cat_encoder.fit_transform(books_feat[['combined_cat']])

# 3. One-hot encode book_type
type_encoder = OneHotEncoder()
book_type_sparse = type_encoder.fit_transform(books_feat[['book_type']].fillna('unknown'))

# 4. Combine features into a final matrix
# Using One-Hot encoding for everything now
tfidf_matrix = hstack([cats_sparse, book_type_sparse]).tocsr()

# 5. Create a mapping from book_id (the mapped integer) to the row index in the matrix
book_idx_to_tfidf_row = {}
for idx, row in books.iterrows():
    orig_id = row['i']
    if orig_id in book_id_map:
        mapped_id = book_id_map[orig_id]
        book_idx_to_tfidf_row[mapped_id] = idx

print(f"Feature matrix shape (One-Hot): {tfidf_matrix.shape}")
print(f"Mapping created for {len(book_idx_to_tfidf_row)} books.")

Feature matrix shape (One-Hot): (15291, 77)
Mapping created for 15109 books.


# TF-IDF with Categories (optimizing the weight of the categories)

In [ ]:
import time
import re

K_FOLDS = 5

# Fill NaN values first!
books['Author'] = books['Author'].fillna('')
books['Subjects'] = books['Subjects'].fillna('')
books['Publisher'] = books['Publisher'].fillna('')
books['Title'] = books['Title'].fillna('')

# Clean text function
def clean_text(text):
    text = re.sub(r'--', ' ', text)
    text = re.sub(r'-', ' ', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

books['Subjects_clean'] = books['Subjects'].apply(clean_text)

# Build TF-IDF function
def build_tfidf(content_series, max_features=10000):
    tfidf = TfidfVectorizer(max_features=max_features, stop_words=None,
                            strip_accents='unicode', min_df=2)
    matrix = tfidf.fit_transform(content_series)
    return matrix

# Now build variants
print("Building TF-IDF variants...")
variants = {}

# Current (baseline)
books['v_current'] = (books['Title'] + ' ' + books['Author'] + ' ' +
                      books['Subjects'] + ' ' + books['Subjects'] + ' ' +
                      books['Publisher'])
variants['current'] = build_tfidf(books['v_current'])

# Subjects x3
books['v_subj_x3'] = (books['Title'] + ' ' + books['Author'] + ' ' +
                       books['Subjects'] + ' ' + books['Subjects'] + ' ' +
                       books['Subjects'])
variants['subj_x3'] = build_tfidf(books['v_subj_x3'])

# Subjects x4
books['v_subj_x4'] = (books['Subjects'] + ' ' + books['Subjects'] + ' ' +
                       books['Subjects'] + ' ' + books['Subjects'] + ' ' +
                       books['Author'])
variants['subj_x4'] = build_tfidf(books['v_subj_x4'])

# Author x2
books['v_author_x2'] = (books['Title'] + ' ' + books['Author'] + ' ' +
                         books['Author'] + ' ' + books['Subjects'] + ' ' +
                         books['Subjects'] + ' ' + books['Publisher'])
variants['author_x2'] = build_tfidf(books['v_author_x2'])

# Subjects only
books['v_subj_only'] = (books['Subjects'] + ' ' + books['Subjects'] + ' ' +
                         books['Subjects'])
variants['subj_only'] = build_tfidf(books['v_subj_only'])

# No Publisher
books['v_no_pub'] = (books['Title'] + ' ' + books['Author'] + ' ' +
                      books['Subjects'] + ' ' + books['Subjects'])
variants['no_pub'] = build_tfidf(books['v_no_pub'])

# Cleaned version — remove punctuation from Subjects
def clean_text(text):
    text = re.sub(r'--', ' ', text)
    text = re.sub(r'-', ' ', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

books['Subjects_clean'] = books['Subjects'].apply(clean_text)

books['v_cleaned'] = (books['Title'] + ' ' + books['Author'] + ' ' +
                      books['Subjects_clean'] + ' ' + books['Subjects_clean'] + ' ' +
                      books['Publisher'])
variants['cleaned'] = build_tfidf(books['v_cleaned'])

print(f"Built {len(variants)} variants!")

# Build mapping for each variant (same for all since books['i'] doesn't change)
mapping = {book_id_map[orig_id]: idx
           for idx, orig_id in enumerate(books['i'])
           if orig_id in book_id_map}

# K-Fold evaluation
scores_per_variant = {v: [] for v in variants}

interactions_sorted = interactions.sort_values(["user_id", "timestamp"]).copy()
interactions_sorted["fold"] = interactions_sorted.groupby("user_id")["timestamp"].transform(
    lambda x: pd.qcut(x.rank(method='first'), K_FOLDS, labels=False)
)

for fold in range(K_FOLDS):
    print(f"\n── Fold {fold+1}/{K_FOLDS} ──")

    train_data = interactions_sorted[interactions_sorted["fold"] != fold]
    test_data = interactions_sorted[interactions_sorted["fold"] == fold]

    train_matrix = create_weighted_matrix(train_data, n_users, n_items, decay=0.03)

    test_matrix = np.zeros((n_users, n_items))
    test_matrix[test_data["user_id"].values, test_data["book_id"].values] = 1

    # CF hybrid
    user_sim = cosine_similarity(train_matrix)
    user_pred = user_based_predict(train_matrix, user_sim)
    del user_sim

    item_sim = cosine_similarity(train_matrix.T)
    item_pred = item_based_predict(train_matrix, item_sim)
    del item_sim

    cf_hybrid = 0.45 * normalize_matrix(user_pred) + 0.55 * normalize_matrix(item_pred)
    del user_pred, item_pred

    # Popularity
    book_popularity = np.array(train_matrix.sum(axis=0)).flatten()
    pop_norm = normalize_matrix(book_popularity)

    # Evaluate each variant
    for variant_name, tfidf_matrix in variants.items():
        content_pred = np.zeros((n_users, n_items))
        for user_idx in range(n_users):
            train_books = train_data[train_data['user_id'] == user_idx]['book_id'].values
            if len(train_books) == 0:
                continue
            rows = [mapping[idx] for idx in train_books if idx in mapping]
            if len(rows) == 0:
                continue
            user_profile = np.asarray(tfidf_matrix[rows].mean(axis=0))
            scores = cosine_similarity(user_profile, tfidf_matrix).flatten()
            for idx, orig_id in enumerate(books['i']):
                if orig_id in book_id_map:
                    content_pred[user_idx, book_id_map[orig_id]] = scores[idx]

        content_norm = normalize_matrix(content_pred)
        del content_pred

        hybrid = 0.75 * cf_hybrid + 0.20 * content_norm + 0.05 * pop_norm
        precision, _ = precision_recall_at_k(hybrid, test_matrix, k=10)
        scores_per_variant[variant_name].append(precision)
        print(f"  {variant_name} | Precision@10: {precision:.4f}")
        del hybrid, content_norm

    del cf_hybrid, pop_norm

# Results
print("\n=== Average Precision@10 across 5 folds ===")
best_variant, best_score = None, 0
for v in variants:
    avg = np.mean(scores_per_variant[v])
    print(f"{v:12s} | Avg Precision@10: {avg:.4f}")
    if avg > best_score:
        best_score = avg
        best_variant = v

print(f"\n✓ Best variant: {best_variant} → Avg Precision@10: {best_score:.4f}")

Building TF-IDF variants...


KeyboardInterrupt: 

Calculating Presicion and Recall:

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import re

# --- 1. METRIC FUNCTIONS ---
def precision_recall_at_k(scores, ground_truth, k=10):
    n_users = scores.shape[0]
    total_p = 0.0
    total_r = 0.0
    valid_users = 0

    for u in range(n_users):
        true_items = np.where(ground_truth[u] == 1)[0]
        # Skip users who didn't interact with any items in the test set
        if len(true_items) == 0:
            continue

        top_k = np.argsort(scores[u])[-k:]
        hits = np.isin(top_k, true_items).sum()

        total_p += hits / k
        total_r += hits / len(true_items)
        valid_users += 1

    return total_p / valid_users, total_r / valid_users if valid_users > 0 else (0, 0)

# --- 2. TEXT PREPARATION (Only the winning variant: author_x2) ---
print("Preparing TF-IDF (Only author_x2 variant)...")
books['Author'] = books['Author'].fillna('')
books['Subjects'] = books['Subjects'].fillna('')
books['Publisher'] = books['Publisher'].fillna('')
books['Title'] = books['Title'].fillna('')

books['v_author_x2'] = (books['Title'] + ' ' + books['Author'] + ' ' +
                         books['Author'] + ' ' + books['Subjects'] + ' ' +
                         books['Subjects'] + ' ' + books['Publisher'])

tfidf = TfidfVectorizer(max_features=10000, stop_words=None, strip_accents='unicode', min_df=2)
tfidf_matrix = tfidf.fit_transform(books['v_author_x2'])

mapping = {book_id_map[orig_id]: idx for idx, orig_id in enumerate(books['i']) if orig_id in book_id_map}

# --- 3. 5-FOLD CROSS-VALIDATION ---
K_FOLDS = 5
interactions_sorted = interactions.sort_values(["user_id", "timestamp"]).copy()
interactions_sorted["fold"] = interactions_sorted.groupby("user_id")["timestamp"].transform(
    lambda x: pd.qcut(x.rank(method='first'), K_FOLDS, labels=False)
)

precisions, recalls = [], []

for fold in range(K_FOLDS):
    print(f"Calculating Fold {fold+1}/{K_FOLDS}...")

    train_data = interactions_sorted[interactions_sorted["fold"] != fold]
    test_data = interactions_sorted[interactions_sorted["fold"] == fold]

    train_matrix = create_weighted_matrix(train_data, n_users, n_items, decay=0.03)
    test_matrix = np.zeros((n_users, n_items))
    test_matrix[test_data["user_id"].values, test_data["book_id"].values] = 1

    # CF predictions
    user_pred = user_based_predict(train_matrix, cosine_similarity(train_matrix))
    item_pred = item_based_predict(train_matrix, cosine_similarity(train_matrix.T))
    cf_hybrid = 0.45 * normalize_matrix(user_pred) + 0.55 * normalize_matrix(item_pred)
    del user_pred, item_pred

    # Popularity prediction
    pop_norm = normalize_matrix(np.array(train_matrix.sum(axis=0)).flatten())

    # Content prediction (author_x2)
    content_pred = np.zeros((n_users, n_items))
    for user_idx in range(n_users):
        train_books = train_data[train_data['user_id'] == user_idx]['book_id'].values
        if len(train_books) == 0: continue
        rows = [mapping[idx] for idx in train_books if idx in mapping]
        if len(rows) == 0: continue
        user_profile = np.asarray(tfidf_matrix[rows].mean(axis=0))
        scores = cosine_similarity(user_profile, tfidf_matrix).flatten()
        for idx, orig_id in enumerate(books['i']):
            if orig_id in book_id_map:
                content_pred[user_idx, book_id_map[orig_id]] = scores[idx]

    content_norm = normalize_matrix(content_pred)
    del content_pred

    # Final hybrid (using your manual weights)
    hybrid = 0.75 * cf_hybrid + 0.20 * content_norm + 0.05 * pop_norm

    # Calculate and store metrics
    p, r = precision_recall_at_k(hybrid, test_matrix, k=10)
    precisions.append(p)
    recalls.append(r)
    print(f"  --> Precision: {p:.4f} | Recall: {r:.4f}")

# --- FINAL OUTPUT FOR THE TABLE ---
print("\n" + "="*40)
print(f"BASELINE MODEL (Manual Tuning) 5-Fold")
print(f"Average Precision@10: {np.mean(precisions):.4f}")
print(f"Average Recall@10:    {np.mean(recalls):.4f}")
print("="*40)

Preparing TF-IDF (Only author_x2 variant)...
Calculating Fold 1/5...
  --> Precision: 0.0595 | Recall: 0.3052
Calculating Fold 2/5...
  --> Precision: 0.0703 | Recall: 0.3601
Calculating Fold 3/5...
  --> Precision: 0.0656 | Recall: 0.3770
Calculating Fold 4/5...
  --> Precision: 0.0736 | Recall: 0.3685
Calculating Fold 5/5...
  --> Precision: 0.0599 | Recall: 0.3225

BASELINE MODEL (Manual Tuning) 5-Fold
Average Precision@10: 0.0658
Average Recall@10:    0.3467


# Now train on full data and save the file with recommendations:

In [ ]:
# ── Final model with author_x2 TF-IDF ────────────────────────────────────

# Step 1: Build author_x2 content
books['Author'] = books['Author'].fillna('')
books['Subjects'] = books['Subjects'].fillna('')
books['Publisher'] = books['Publisher'].fillna('')
books['Title'] = books['Title'].fillna('')

books['v_author_x2'] = (books['Title'] + ' ' +
                         books['Author'] + ' ' + books['Author'] + ' ' +
                         books['Subjects'] + ' ' + books['Subjects'] + ' ' +
                         books['Publisher'])

tfidf_author_x2 = TfidfVectorizer(max_features=10000, stop_words=None,
                                    strip_accents='unicode', min_df=2)
tfidf_matrix_author_x2 = tfidf_author_x2.fit_transform(books['v_author_x2'])

mapping_author_x2 = {book_id_map[orig_id]: idx
                      for idx, orig_id in enumerate(books['i'])
                      if orig_id in book_id_map}
print(f"TF-IDF author_x2 shape: {tfidf_matrix_author_x2.shape}")

# Step 2: Full weighted matrix
full_matrix = create_weighted_matrix(interactions, n_users, n_items, decay=0.03)

# Step 3: CF hybrid
user_sim_full = cosine_similarity(full_matrix)
user_pred_full = user_based_predict(full_matrix, user_sim_full)
del user_sim_full

item_sim_full = cosine_similarity(full_matrix.T)
item_pred_full = item_based_predict(full_matrix, item_sim_full)
del item_sim_full

cf_hybrid = 0.45 * normalize_matrix(user_pred_full) + 0.55 * normalize_matrix(item_pred_full)
del user_pred_full, item_pred_full
print("CF hybrid ready!")

# Step 4: Content with author_x2
content_pred = np.zeros((n_users, n_items))
for user_idx in range(n_users):
    read_indices = np.where(full_matrix[user_idx, :] > 0)[0]
    if len(read_indices) == 0:
        continue
    rows = [mapping_author_x2[idx] for idx in read_indices
            if idx in mapping_author_x2]
    if len(rows) == 0:
        continue
    user_profile = np.asarray(tfidf_matrix_author_x2[rows].mean(axis=0))
    scores = cosine_similarity(user_profile, tfidf_matrix_author_x2).flatten()
    for idx, orig_id in enumerate(books['i']):
        if orig_id in book_id_map:
            content_pred[user_idx, book_id_map[orig_id]] = scores[idx]

content_norm = normalize_matrix(content_pred)
del content_pred
print("Content ready!")

# Step 5: Popularity
full_binary = create_data_matrix(interactions, n_users, n_items)
book_popularity = np.array(full_binary.sum(axis=0)).flatten()
pop_norm = normalize_matrix(book_popularity)

# Step 6: Final hybrid
hybrid_final = 0.75 * cf_hybrid + 0.20 * content_norm + 0.05 * pop_norm
del cf_hybrid, content_norm, pop_norm
print("Final hybrid ready!")

# Step 7: Save submission
K = 10
with open("submission_author_x2.csv", "w") as f:
    f.write("user_id,recommendation\n")
    for user_idx in range(hybrid_final.shape[0]):
        scores = hybrid_final[user_idx, :].copy()
        top_k_idx = np.argsort(scores)[::-1][:K]
        original_book_ids = [str(book_id_inverse_map[idx]) for idx in top_k_idx]
        original_user_id = user_id_inverse_map[user_idx]
        f.write(f"{original_user_id},{' '.join(original_book_ids)}\n")

with open("submission_author_x2.csv", "r") as f:
    for i, line in enumerate(f):
        print(line.strip())
        if i >= 4: break

TF-IDF author_x2 shape: (15291, 10000)
CF hybrid ready!
Content ready!
Final hybrid ready!
user_id,recommendation
4456,13807 13805 9306 3333 8581 9208 14120 12689 13806 13804
142,1973 1970 1972 1959 1974 806 1568 1962 1975 1977
362,4248 4257 4249 4259 4252 1418 4256 4258 2107 4236
1809,8971 11317 8937 6338 5864 7496 5045 8653 6024 9249
